## 1. Data Loading

We load the SMS Spam Collection dataset from a public GitHub repository. Each message is labeled either **ham** (legitimate) or **spam**. The labels are then mapped to binary integers: `0` for ham and `1` for spam.

In [ ]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv', sep='\t', header=None, names=['label', 'message'])
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

## 2. Text Vectorization (TF-IDF)

Raw text cannot be fed directly into a neural network. We use **TF-IDF** (Term Frequency–Inverse Document Frequency) to convert each message into a numerical vector. Each dimension represents the importance of a word relative to the whole corpus.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['message']).toarray()
y = df['label'].values

print(X.shape)

## 3. Conversion to PyTorch Tensors

We convert the NumPy arrays produced by TF-IDF into **PyTorch tensors** (`float32`). This is required before passing the data to a PyTorch model.

In [ ]:
import torch

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(df['label'].values, dtype=torch.float32)

print(X.shape)
print(y.shape)

## 4. Train / Test Split

We split the dataset into a **training set (80%)** and a **test set (20%)** using a fixed random seed for reproducibility. The model will only learn from the training set; the test set is kept aside to evaluate generalization.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y,
    train_size=0.80,
    test_size=0.20,
    random_state=42)

print(X_train.shape)
print(X_test.shape)

## 5. Model Definition

We define a simple **feed-forward neural network** with three fully connected layers:
- **fc1**: input → 64 neurons  
- **fc2**: 64 → 32 neurons  
- **fc3**: 32 → 1 neuron (binary output)

ReLU activations are used between layers. The final layer outputs a raw logit (no sigmoid here — that is handled by the loss function).

In [ ]:
import torch.nn as nn

class SpamClassifier(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = SpamClassifier(input_size=X_train.shape[1])
print(model)

## 6. Training Loop

We use **BCEWithLogitsLoss** (binary cross-entropy with built-in sigmoid) as the loss function, and **Adam** as the optimizer with a learning rate of `0.001`. The model is trained for 100 epochs over the full training set. The loss is printed every 5 epochs to monitor convergence.

In [ ]:
import torch.optim as optim

# Loss et optimizer
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    predictions = model(X_train)
    loss = loss_fn(predictions, y_train.reshape(-1, 1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

## 7. Evaluation

We switch the model to **eval mode** (disables dropout/batchnorm if any) and run inference on the test set without computing gradients. Predictions are thresholded at `0.5` after applying sigmoid. A **classification report** is printed showing precision, recall, and F1-score for both ham and spam classes.

In [ ]:
from sklearn.metrics import classification_report

model.eval()
with torch.no_grad():
    predictions = model(X_test)
    predicted_labels = (torch.sigmoid(predictions) > 0.5).float()

print(classification_report(y_test, predicted_labels))

## 8. Inference on a New Message

We test the trained model on a custom message. The raw text is vectorized using the **same TF-IDF vectorizer** fitted on the training data, then passed through the model. The output probability is thresholded at `0.5` to produce a final **HAM** or **SPAM** label.

In [ ]:
new_message = ["""Congratulations! You have been selected as a winner of our exclusive lottery. 
You have won $1,000,000! To claim your prize, click the link below and enter 
your bank details. This offer expires in 24 hours. Act now!
www.totally-not-a-scam.com"""]


new_vector = torch.tensor(vectorizer.transform(new_message).toarray(), dtype=torch.float32)


model.eval()
with torch.no_grad():
    output = torch.sigmoid(model(new_vector))
    label = "SPAM" if output > 0.5 else "HAM"
    print(f"Prediction: {label} ({output.item():.2f})")

## 9. Save Model & Vectorizer

We persist the trained model weights and the fitted TF-IDF vectorizer to disk. These files are used by `train.py` / `main.py` in the Docker approach to serve the Streamlit app without retraining.

In [ ]:
import pickle

torch.save(model.state_dict(), 'model.pth')

with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Saved model.pth and vectorizer.pkl")